# Лабораторная работа 8

Задачи:
- Взять предобученную модель, изменить классификатор и пропустить через EuroSAT

In [98]:
!gdown https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip

n
Downloading...
From: https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip
To: /content/EuroSAT_RGB.zip
100% 94.7M/94.7M [01:20<00:00, 1.17MB/s]


In [99]:
!unzip -q EuroSAT_RGB.zip

replace EuroSAT_RGB/Forest/Forest_864.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


In [100]:
import pandas as pd
import numpy as np
import torch
import os
import torchvision
from torchvision import transforms
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import random_split, DataLoader

> ImageNet consists of images in RGB format with varying resolutions. For example, in ImageNet 2012, "fish" category, the resolution ranges from 4288 x 2848 to 75 x 56. In machine learning, these are typically preprocessed into a standard constant resolution, and whitened, before further processing by neural networks.

> For example, in PyTorch, ImageNet images are by default normalized by dividing the pixel values so that they fall between 0 and 1, then subtracting by [0.485, 0.456, 0.406], then dividing by [0.229, 0.224, 0.225]. These are the mean and standard deviations for ImageNet, so this whitens the input data

Изменяем размер входной картинки под ImageNet



In [101]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

batch_size = 128

EuroSAT = torchvision.datasets.ImageFolder(root="/content/EuroSAT_RGB", transform=transform,)

train_size = int(0.8 * len(EuroSAT))
test_size  = len(EuroSAT) - train_size

train_dataset, test_dataset = random_split(EuroSAT, [train_size, test_size])

Train, Test = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size), torch.utils.data.DataLoader(test_dataset, batch_size=batch_size)
ShowCase = iter(Test)
classes = EuroSAT.classes
len(Train)

169

[Отсюда](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html) возьму основной пайплайн

In [121]:
import plotly.express as px

# functions to show an image
def denormalize(img):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    return img * std + mean

def imshow(img, label, pred_label="NoPred", prob=0):
    img = torch.clamp(denormalize(img).permute(1, 2, 0), 0, 1)
    fig = px.imshow(img)

    fig.add_annotation(
    text=f"Class: {label} || Pred: {pred_label} || Prob: {prob:.3f}",
    x=0.5, y=1,
    xref="paper", yref="paper",
    font=dict(color="black", size=16),
)
    fig.update_layout(
        height=400,
        width=600
    )
    fig.show()


# get some random training images
dataiter = iter(Train)
images, labels = next(dataiter)
# show images
imshow(torchvision.utils.make_grid(images[0]), classes[labels[0]])
# print labels
print(images.shape)

torch.Size([128, 3, 224, 224])


# Выбор модели ResNet18

In [103]:
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
model.fc

Linear(in_features=512, out_features=1000, bias=True)

## Замораживаем слои и обучаем новый классификатор

In [104]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [105]:
for param in model.parameters():
    param.requires_grad = False

Из [статьи](https://docs.pytorch.org/vision/main/models.html) по конкретной модели узнаем архитектуру модели и меняем последний слой

In [106]:
model.fc = nn.Sequential(
    nn.Linear(512, len(classes)),
)

In [107]:
def train(model, optimizer, criterion, dataloader, epochs=10, device=device):
  for epoch in range(epochs):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(dataloader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        inputs = inputs.to(device)
        labels = labels.to(device)
        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 10 == 0:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 10:.3f}')
            running_loss = 0.0

  print('Finished Training')

In [108]:
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

train(model, optimizer, criterion, Train, epochs=2)

[1,     1] loss: 0.246
[1,    11] loss: 2.135
[1,    21] loss: 1.530
[1,    31] loss: 1.162
[1,    41] loss: 0.950
[1,    51] loss: 0.818
[1,    61] loss: 0.738
[1,    71] loss: 0.666
[1,    81] loss: 0.606
[1,    91] loss: 0.564
[1,   101] loss: 0.522
[1,   111] loss: 0.495
[1,   121] loss: 0.488
[1,   131] loss: 0.480
[1,   141] loss: 0.435
[1,   151] loss: 0.449
[1,   161] loss: 0.432
[2,     1] loss: 0.045
[2,    11] loss: 0.419
[2,    21] loss: 0.361
[2,    31] loss: 0.352
[2,    41] loss: 0.364
[2,    51] loss: 0.380
[2,    61] loss: 0.378
[2,    71] loss: 0.351
[2,    81] loss: 0.337
[2,    91] loss: 0.341
[2,   101] loss: 0.319
[2,   111] loss: 0.319
[2,   121] loss: 0.338
[2,   131] loss: 0.333
[2,   141] loss: 0.304
[2,   151] loss: 0.321
[2,   161] loss: 0.317
Finished Training


In [109]:
from sklearn.metrics import classification_report

In [110]:
def test(model, test_loader, classes, device=device):
  model.eval()
  y_pred = []
  y_true = []

  with torch.no_grad():
      for inputs, labels in test_loader:
          inputs = inputs.to(device)
          labels = labels.to(device)

          outputs = model(inputs)
          preds = torch.argmax(outputs, dim=1)

          y_pred.extend(preds.cpu().numpy())
          y_true.extend(labels.cpu().numpy())

  print(classification_report(y_true, y_pred, target_names=classes))

In [111]:
test(model, Test, classes)

                      precision    recall  f1-score   support

          AnnualCrop       0.94      0.89      0.92       595
              Forest       0.96      0.98      0.97       582
HerbaceousVegetation       0.94      0.85      0.89       645
             Highway       0.86      0.76      0.81       493
          Industrial       0.91      0.96      0.94       495
             Pasture       0.87      0.91      0.89       415
       PermanentCrop       0.77      0.93      0.85       475
         Residential       0.96      0.96      0.96       598
               River       0.80      0.80      0.80       516
             SeaLake       0.97      0.94      0.96       586

            accuracy                           0.90      5400
           macro avg       0.90      0.90      0.90      5400
        weighted avg       0.90      0.90      0.90      5400



In [112]:
import random

In [113]:
def draw(model, n=3):
  model.to("cpu")
  images, labels = next(ShowCase)

  idx = random.sample(range(len(images)), n)

  images, labels = images[idx], labels[idx]

  probs = torch.softmax(model(images), dim=1)
  pred_label = torch.argmax(probs, dim=1)
  for i in range(n):
    imshow(images[i], classes[labels[i]], classes[pred_label[i]], probs[i][pred_label[i]])

In [114]:
draw(model)

# Выбор модели VGG16

In [115]:
model = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1)
model.classifier

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=1000, bias=True)
)

## Замораживаем слои и обучаем новый классификатор

In [116]:
for param in model.parameters():
    param.requires_grad = False

Из [статьи](https://docs.pytorch.org/vision/main/models.html) по конкретной модели узнаем архитектуру модели и меняем последний слой

In [117]:
model.classifier[6] = nn.Linear(4096, len(classes))

In [118]:
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

train(model, optimizer, criterion, Train, epochs=2)

[1,     1] loss: 0.234
[1,    11] loss: 1.381
[1,    21] loss: 0.665
[1,    31] loss: 0.497
[1,    41] loss: 0.431
[1,    51] loss: 0.397
[1,    61] loss: 0.408
[1,    71] loss: 0.363
[1,    81] loss: 0.343
[1,    91] loss: 0.331
[1,   101] loss: 0.311
[1,   111] loss: 0.312
[1,   121] loss: 0.334
[1,   131] loss: 0.312
[1,   141] loss: 0.276
[1,   151] loss: 0.297
[1,   161] loss: 0.310
[2,     1] loss: 0.039
[2,    11] loss: 0.288
[2,    21] loss: 0.272
[2,    31] loss: 0.265
[2,    41] loss: 0.276
[2,    51] loss: 0.258
[2,    61] loss: 0.292
[2,    71] loss: 0.277
[2,    81] loss: 0.256
[2,    91] loss: 0.263
[2,   101] loss: 0.243
[2,   111] loss: 0.256
[2,   121] loss: 0.287
[2,   131] loss: 0.263
[2,   141] loss: 0.232
[2,   151] loss: 0.260
[2,   161] loss: 0.255
Finished Training


In [119]:
test(model, Test, classes)

                      precision    recall  f1-score   support

          AnnualCrop       0.96      0.91      0.94       595
              Forest       0.94      0.99      0.96       582
HerbaceousVegetation       0.95      0.91      0.93       645
             Highway       0.91      0.77      0.83       493
          Industrial       0.96      0.94      0.95       495
             Pasture       0.86      0.93      0.89       415
       PermanentCrop       0.87      0.93      0.90       475
         Residential       0.94      0.96      0.95       598
               River       0.81      0.89      0.85       516
             SeaLake       0.98      0.96      0.97       586

            accuracy                           0.92      5400
           macro avg       0.92      0.92      0.92      5400
        weighted avg       0.92      0.92      0.92      5400



In [120]:
draw(model)

# Итоги

Предобученные модели на ImageNet показали себя очень хорошо с минимальным f1 - 0.8. Основная путаница в highway и river, пока непонимаю связано ли это с преобразованием входных изображений с 64x64 до 224x224 или нет.